# SeaDronesSee v2 — Faster R-CNN ConvNeXt-FPN Selective QAT
Pipeline Colab tiết kiệm tài nguyên: mỗi lần gọi chỉ train **đúng 1 epoch**, validation, benchmark 100 ảnh, lưu checkpoint lên Drive rồi thoát. Chạy lại cell ở phiên Colab tiếp theo để tự resume. Chọn **Runtime → Change runtime type → GPU** trước khi chạy.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước'
print(torch.cuda.get_device_name(0))
print('GPU memory GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)

## 1. Lấy source code
Commit/push các thay đổi pipeline lên GitHub trước khi chạy notebook này.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/EchteAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)
print('Repository:', Path.cwd())

In [ ]:
!pip -q install -e '.[coco]' psutil
!apt-get -qq update && apt-get -qq install -y rclone
!python -m compileall -q Python/EchteAI/pipelines/convnext_qat scripts

## 2. Tải dataset vào SSD tạm của Colab
Không đặt dataset 10 GB trên Drive vì đọc hàng nghìn ảnh qua Drive rất chậm. Checkpoint vẫn được ghi vào Drive. `rclone copy` có thể chạy lại và sẽ tiếp tục các file còn thiếu.

In [ ]:
DATA_ROOT = Path('/content/seadronessee')
DAV_URL = 'https://cloud.cs.uni-tuebingen.de/public.php/dav/files/ZZxX65FGnQ8zjBP/'
subprocess.run(['rclone', 'config', 'delete', 'seadrones'], check=False, capture_output=True)
subprocess.run(['rclone', 'config', 'create', 'seadrones', 'webdav', 'url', DAV_URL, 'vendor', 'nextcloud'], check=True)
subprocess.run([
    'rclone', 'copy', 'seadrones:Compressed Version', str(DATA_ROOT),
    '--progress', '--transfers', '8', '--checkers', '16', '--retries', '10',
], check=True)
required = [
    DATA_ROOT/'images/train', DATA_ROOT/'images/val', DATA_ROOT/'images/test',
    DATA_ROOT/'annotations/instances_train.json',
    DATA_ROOT/'annotations/instances_val.json',
]
assert all(path.exists() for path in required), [str(path) for path in required if not path.exists()]
print('Dataset ready at', DATA_ROOT)

In [ ]:
import json, collections
for split in ('train', 'val'):
    data = json.loads((DATA_ROOT/f'annotations/instances_{split}.json').read_text())
    counts = collections.Counter(a['category_id'] for a in data['annotations'])
    names = {c['id']: c['name'] for c in data['categories']}
    print(split, 'images=', len(data['images']), 'boxes=', len(data['annotations']))
    print({names[key]: value for key, value in counts.items()})

## 3. Smoke test

In [ ]:
!python tests/smoke_selective_qat.py

## 4. Train đúng 1 epoch FP32
Mỗi lần chạy cell này, `train_next_epoch.py` đọc `fp32_last.pt`, train đúng epoch kế tiếp, validation + benchmark 100 ảnh, xác minh checkpoint trên Drive rồi thoát. FP32 dùng batch size 2 và giữ nguyên kích thước ảnh. Chạy lại cell trong phiên mới cho đến 30/30.

In [ ]:
import datetime

def run_and_log(command, log_path):
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Command:', ' '.join(command), flush=True)
    print('Persistent log:', log_path, flush=True)
    print('Started:', datetime.datetime.now().isoformat(timespec='seconds'), flush=True)
    with log_path.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== START {datetime.datetime.now().isoformat()} =====\n')
        log_file.write(' '.join(command) + '\n')
        log_file.flush()
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=env,
        )
        for line in process.stdout:
            print(line, end='', flush=True)
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
        log_file.write(f'===== END code={return_code} {datetime.datetime.now().isoformat()} =====\n')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

CONFIG = 'configs/seadronessee_colab.yaml'
DRIVE_OUT = Path('/content/drive/MyDrive/EchteAI/seadronessee_m3')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', 'scripts/train_next_epoch.py',
    '--config', CONFIG, '--stage', 'fp32',
]
run_and_log(command, DRIVE_OUT/'fp32_train.log')

## 5. Train đúng 1 epoch Selective QAT M3
Chỉ bắt đầu sau khi FP32 đạt 30/30. Mỗi lần chạy cell này chỉ thực hiện 1 epoch QAT với batch size 1. Lần đầu calibration 512 ảnh; checkpoint giữ cả optimizer, observer và fake-quant để phiên sau nối đúng pha: weight-only 2 epoch → full QAT 6 epoch → frozen observer 3 epoch. Epoch 11 tự convert INT8.

In [ ]:
command = [
    sys.executable, '-u', 'scripts/train_next_epoch.py',
    '--config', CONFIG, '--stage', 'qat', '--variant', 'M3',
]
run_and_log(command, DRIVE_OUT/'qat_train.log')

## 6. Kiểm tra tiến độ checkpoint trên Drive

In [ ]:
def checkpoint_epoch(path):
    path = Path(path)
    if not path.exists():
        return 0
    return int(torch.load(path, map_location='cpu', weights_only=False).get('epoch', 0))

fp32_epoch = checkpoint_epoch(DRIVE_OUT/'fp32_last.pt')
qat_epoch = checkpoint_epoch(DRIVE_OUT/'qat_last.pt')
print(f'FP32 progress: {fp32_epoch}/30')
print(f'QAT progress:  {qat_epoch}/11')
print('FP32 best:', (DRIVE_OUT/'fp32_best.pt').exists())
print('QAT best:', (DRIVE_OUT/'qat_best.pt').exists())
print('INT8 ready:', (DRIVE_OUT/'selective_int8.pt').exists())
print('Epoch benchmark history:', DRIVE_OUT/'epoch_benchmarks.json')

## 7. Evaluate trên validation
Test annotation chính thức bị giữ kín, vì vậy metric local được tính trên validation.

In [ ]:
fp32_best = DRIVE_OUT/'fp32_best.pt'
int8_model = DRIVE_OUT/'selective_int8.pt'
assert fp32_best.exists(), 'Chưa có fp32_best.pt'
assert int8_model.exists(), 'QAT chưa hoàn tất 11/11 nên chưa có selective_int8.pt'
subprocess.run([sys.executable, '-u', 'scripts/evaluate.py', '--config', CONFIG, '--model', 'fp32', '--split', 'val'], check=True)
subprocess.run([sys.executable, '-u', 'scripts/evaluate.py', '--config', CONFIG, '--model', 'int8', '--split', 'val'], check=True)

## 8. So sánh FP32–INT8 công bằng trên CPU và kiểm tra Drive
Cả hai model chạy cùng CPU, batch size 1 và cùng 100 ảnh. Báo cáo kích thước backbone/full model, accuracy, precision, mean IoU, mAP, latency và speedup.

In [ ]:
command = [
    sys.executable, '-u', 'scripts/compare_fp32_int8.py',
    '--config', CONFIG,
    '--fp32-checkpoint', str(DRIVE_OUT/'fp32_best.pt'),
    '--int8-checkpoint', str(DRIVE_OUT/'selective_int8.pt'),
    '--images', '100', '--threads', '1',
    '--output', str(DRIVE_OUT/'fp32_int8_comparison.json'),
]
run_and_log(command, DRIVE_OUT/'fp32_int8_comparison.log')
for path in sorted(DRIVE_OUT.glob('*')):
    print(f'{path.name:28s} {path.stat().st_size / 2**20:9.2f} MB')